# CMySE 911 · Modelo de pronóstico de la demanda

Reproduce el modelo del capítulo 5 de la tesis: **SARIMAX(2,1,0)(1,1,1)[12]** sobre totales
mensuales de incidentes, 2018-2025.

**Qué hace:** estima la retícula de 143 especificaciones, las evalúa sobre 2025 fuera de muestra,
selecciona la de menor error entre las que el AIC no distingue, reajusta sobre los 96 meses y
proyecta 2026-2027 con intervalo de confianza al 95 %.

**Cómo se usa:** ejecutar las celdas en orden. La primera pide el archivo `serie_mensual.csv`, que
está en la carpeta `02_modelo` del repositorio. Tarda unos 3 a 5 minutos.

In [ ]:
!pip install -q statsmodels

import io, os, itertools, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")

if not os.path.exists("serie_mensual.csv"):
    from google.colab import files
    cargados = files.upload()

p = pd.read_csv("serie_mensual.csv", encoding="utf-8-sig")
y = np.asarray(p["Incidentes"], float)
print(f"serie: {len(y)} meses, de {p['Periodo'].iloc[0]} a {p['Periodo'].iloc[-1]}")
print(f"total del período: {y.sum():,.0f} incidentes")

## 1. Protocolo de selección

Entrenamiento con 2018-01 a 2024-12 (84 meses) y validación con 2025 completo (12 meses fuera de
muestra). Se estiman todas las combinaciones con p,q ∈ 0..2, d ∈ {0,1}, P,Q ∈ {0,1}, D ∈ {0,1} y
período 12. Se descartan los modelos que no convergen.

In [ ]:
N_TEST = 12
train, test = y[:-N_TEST], y[-N_TEST:]

def ajustar(datos, order, sorder):
    return SARIMAX(datos, order=order, seasonal_order=sorder,
                   enforce_stationarity=True, enforce_invertibility=True
                   ).fit(disp=False, maxiter=200)

modelos, fallos = [], 0
for p_, d_, q_, P_, D_, Q_ in itertools.product(range(3), [0, 1], range(3), range(2), [0, 1], range(2)):
    if (p_, d_, q_) == (0, 0, 0) and (P_, D_, Q_) == (0, 0, 0):
        continue
    try:
        f = ajustar(train, (p_, d_, q_), (P_, D_, Q_, 12))
        modelos.append({"orden": (p_, d_, q_), "estacional": (P_, D_, Q_, 12),
                        "aic": f.aic, "bic": f.bic})
    except Exception:
        fallos += 1

modelos.sort(key=lambda r: r["aic"])
print(f"especificaciones estimadas: {len(modelos)} (fallos: {fallos})")
print("mejor por AIC:", modelos[0]["orden"], modelos[0]["estacional"], f"AIC {modelos[0]['aic']:.3f}")

In [ ]:
# validacion fuera de muestra de los 8 mejores por AIC
def metricas(real, pred):
    e = np.asarray(real) - np.asarray(pred)
    return {"MAE": np.mean(np.abs(e)), "RMSE": np.sqrt(np.mean(e ** 2)),
            "MAPE": np.mean(np.abs(e / real)) * 100, "sesgo": np.mean(e)}

for r in modelos[:8]:
    f = ajustar(train, r["orden"], r["estacional"])
    r["test"] = metricas(test, f.get_forecast(steps=N_TEST).predicted_mean)

# entre los indistinguibles por AIC (delta <= 2), el de menor RMSE
candidatos = [r for r in modelos[:8] if r["aic"] - modelos[0]["aic"] <= 2.0]
elegido = min(candidatos, key=lambda r: r["test"]["RMSE"])

print(f"candidatos con delta AIC <= 2: {len(candidatos)}")
tabla = pd.DataFrame([{"orden": str(r["orden"]), "estacional": str(r["estacional"]),
                      "AIC": round(r["aic"], 3),
                      "MAE": round(r["test"]["MAE"], 2), "RMSE": round(r["test"]["RMSE"], 2),
                      "MAPE": round(r["test"]["MAPE"], 3)} for r in candidatos])
print(tabla.to_string(index=False))
print(f"\nelegido: SARIMAX{elegido['orden']}x{elegido['estacional']}")

## 2. Desempeño frente a las líneas base

El error absoluto medio escalado (MASE) compara el modelo con el ingenuo estacional sobre el mismo
tramo. Un valor menor que 1 indica mejora sobre la referencia.

In [ ]:
n_ultimo = metricas(test, np.repeat(train[-1], N_TEST))
n_estac = metricas(test, (list(train[-12:]) * 2)[:N_TEST])
t = elegido["test"]
mase = t["MAE"] / n_estac["MAE"]

print(f"{'metrica':8}{'modelo':>12}{'ingenuo estacional':>20}{'ingenuo ultimo':>16}")
for k in ("MAE", "RMSE", "MAPE", "sesgo"):
    print(f"{k:8}{t[k]:>12.2f}{n_estac[k]:>20.2f}{n_ultimo[k]:>16.2f}")
print(f"\nMASE: {mase:.3f}")
print(f"reduccion del RMSE frente al ingenuo estacional: {100*(1-t['RMSE']/n_estac['RMSE']):.1f} %")

## 3. Reajuste y proyección

El modelo se reestima con los 96 meses y se proyectan 24. El intervalo es analítico, condicional en
los parámetros estimados.

In [ ]:
final = ajustar(y, elegido["orden"], elegido["estacional"])
fc = final.get_forecast(steps=24)
pron = np.asarray(fc.predicted_mean, float)
ic = np.asarray(fc.conf_int(alpha=0.05), float)

total_2026 = pron[:12].sum()
total_2027 = pron[12:].sum()
print(f"proyeccion 2026: {total_2026:,.0f} incidentes")
print(f"proyeccion 2027: {total_2027:,.0f} incidentes")
print(f"variacion 2025 -> 2026: {100*(total_2026/y[-12:].sum()-1):.1f} %")

lb = acorr_ljungbox(final.resid, lags=[12], return_df=True)
print(f"\nLjung-Box a 12 rezagos: estadistico {lb['lb_stat'].iloc[0]:.3f}, p = {lb['lb_pvalue'].iloc[0]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
idx = np.arange(len(y))
ax.plot(idx, y, color="#1f3864", lw=1.6, label="Observado 2018-2025")
xi = np.arange(len(y) - 1, len(y) + 24)
ax.plot(xi[1:], pron, color="#c00000", lw=1.8, label="Proyeccion SARIMAX")
ax.fill_between(xi[1:], ic[:, 0], ic[:, 1], color="#c00000", alpha=0.15, label="IC 95 %")
ax.axvline(len(y) - 1, color="grey", ls="--", lw=0.9)
ax.set_title("Serie mensual y proyeccion 2026-2027", fontweight="bold", color="#1f3864")
ax.set_ylabel("incidentes por mes")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

mensual = pd.DataFrame({"mes": [f"2026-{m:02d}" for m in range(1, 13)],
                       "pronostico": pron[:12].round(0),
                       "li95": ic[:12, 0].round(0), "ls95": ic[:12, 1].round(0)})
print(mensual.to_string(index=False))

## 4. Qué debería dar

Con el archivo incluido en el repositorio, la corrida reproduce los valores del capítulo 5:

| Métrica | Valor esperado |
|---|---|
| Especificaciones estimadas | 143 |
| Modelo elegido | SARIMAX(2,1,0)(1,1,1)[12] |
| MAE de prueba | 162,76 |
| RMSE de prueba | 213,30 |
| MAPE de prueba | 1,96 % |
| MASE | 0,20 |
| Proyección 2026 | 94.380 |
| Ljung-Box (12) | 17,591 · p = 0,129 |

**Sobre el intervalo anual.** Los intervalos de la última tabla son mensuales. El del total anual no
se obtiene sumándolos, porque los doce errores comparten el componente de nivel: la tesis lo calcula
por simulación de 20.000 trayectorias y obtiene 50.313 a 137.685.